In [ ]:
# installation des bibliothèques pré-requises
!pip install rasterio shapely geopandas earthpy cartopy

In [ ]:
# pré-requis

import shutil

import os
import glob as glob
import numpy as np
import rasterio as rio
from rasterio import plot
from rasterio.plot import show
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.cm import get_cmap
from matplotlib.axes import Axes
from IPython.display import HTML
from skimage import exposure

from rasterio.mask import mask
from shapely.geometry import mapping
import matplotlib.pyplot as plt
import geopandas as gpd
import earthpy as et
import earthpy.plot as ep
import earthpy.spatial as es
import cartopy as cp

import pandas as pd

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform

In [ ]:
REPERTOIRE_DATASET = "D:\\DATA\\FOLDER"

REPERTOIRE_SATELLITE = "Planet_50_km2" # TOUTES LES IMAGES SONT INCLUSES, ALIBORI COMPRIS
#REPERTOIRE_SATELLITE = "Last_Planet.KO-OK"

EXTENSION_IMAGES_SAT = ".tif"
REPERTOIRE_COMPLET_SATELLITE =  REPERTOIRE_DATASET + "\\" + REPERTOIRE_SATELLITE
REPERTOIRE_SATELLITE_DECOUPE = "Planet_DECOUPE_FINAL_TEST"
#REPERTOIRE_SATELLITE_DECOUPE = "Planet_DECOUPE"
REPERTOIRE_COMPLET_SATELLITE_DECOUPE = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SATELLITE_DECOUPE
print("Nom satellite")
print(REPERTOIRE_SATELLITE)
print("Extension image satellite")
print(EXTENSION_IMAGES_SAT)
print("Répertoire complet satellite")
print(REPERTOIRE_COMPLET_SATELLITE)
print("Répertoire complet satellite découpé")
print(REPERTOIRE_COMPLET_SATELLITE_DECOUPE)

REPERTOIRE_SHAPES = "ROI_3ha"
REPERTOIRE_COMPLET_SHAPES = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SHAPES

Découpages des images suivant les ROI

In [ ]:
# création du dossier de destination
if not os.path.exists(REPERTOIRE_COMPLET_SATELLITE_DECOUPE):
    os.mkdir(REPERTOIRE_COMPLET_SATELLITE_DECOUPE)

listes_rep = os.listdir(REPERTOIRE_COMPLET_SATELLITE)
num_obs = 0
for rep in listes_rep:
    # récupération de l'ID
    split = os.path.splitext(rep)
    nom_fichier = split[0]
    extension = split[1]
    nom_fichier_decoupe = nom_fichier.split("_")
    id = nom_fichier_decoupe[0]

    print("Observation en cours : ", str(id))

    # récupération de la liste des images de l'obs id
    dossier_id_source = REPERTOIRE_COMPLET_SATELLITE + '\\' + rep + '\\' + 'PSScene'
    liste_images_sat = os.listdir(dossier_id_source)

    # création du dossier id dans le repertoire des images découpées
    dossier_id_dest = REPERTOIRE_COMPLET_SATELLITE_DECOUPE + "\\" + str(id)
    if not os.path.exists(dossier_id_dest):
        os.mkdir(dossier_id_dest)

    # découpage des images
    for image in liste_images_sat:
        split = os.path.splitext(image)
        extension = split[1]
        if extension == EXTENSION_IMAGES_SAT: # pour ne pas considérer les fichiers non image
            #print(image)
            # Filtrer image
            fichier_img = image.split("_")
            caracteristique_img = nom_fichier_decoupe[-2]
            if caracteristique_img == 'sr':
                #lecture du fichier image
                chemin_image = dossier_id_source + "\\" + image
                src = rio.open(chemin_image)
                
                #lecture des shp
                shp = gpd.read_file(REPERTOIRE_COMPLET_SHAPES + "\\" + str(id) + '.shp')
    
                # projection du shapefile en concordance avec l'image satellite
                profil_crs_img = src.profile
                shp_utm31N = shp.to_crs(profil_crs_img["crs"])
                shape_reproj = REPERTOIRE_COMPLET_SHAPES + "\\" + str(id) + "_reproj.shp"
                shp_utm31N.to_file(shape_reproj, mode="w")
                #découpage suivant le shp
                img_decoupee, metadonnees = es.crop_image(src, shp_utm31N)
                
                # mise à jour des métadonnées
                # img_decoupee.shape #  (bands, rows, columns)
                metadonnees.update({"driver": "GTiff",
                                 "height": img_decoupee.shape[1],
                                 "width": img_decoupee.shape[2],
                                 "transform": metadonnees["transform"]})
            
                # Save to disk so you can use the file later.
                chemin_image_decoupee = dossier_id_dest + "\\" + image
                with rio.open(chemin_image_decoupee, 'w', **metadonnees) as ff:
                    ff.write(img_decoupee)
    num_obs += 1

print("Nombre d'observations traitées au total", num_obs)

In [ ]:
with rio.open(chemin_image_decoupee, 'w', **metadonnees) as ff:
    ff.write(img_decoupee)

In [ ]:
ff.close()